In [1]:
import pandas as pd
import os
import re

**Data Aquisition:**

Voting Sessions: IDs vindo dos datasets votacoes-{ano}.csv

Voting Propositions Text: Descrição breve do assunto votado vindo da coluna 'ultimaAberturaVotacao_descricao' dos datasets 'votacoes-{ano}.csv'

Voting Propositions Theme: coluna 'tema' vindo dos datasets proposicoesTemas-{ano}.csv

Authors: coluna 'author' vindo do datafrrme 'df_sessions_author.csv' ou proposicoesAutores-{ano}.csv

Votes Recommendations: vindo ddos datasets 'votacoesOrientacoes-{year}.csv

Voters: vindo dos datasets votacoesVotos-{year}.csv

Votes: vindo dos datasets votacoesVotos-{year}.csv

Approval: vindo da coluna 'aprovado' dos datasets votacoes-{ano}.csv


**Feature Engineering:**

Author Popularity:

Propositions Theme based Cluster:

Voters Community:

Propositions Voting based Cluster:

Community Preference:

Label:

In [2]:
# Define the path pattern
base_path = "../data/voting/"
years = range(2003, 2025)

In [3]:
# Initialize an empty list to store dataframes
dfs = []

# Loop through years and load each CSV
for year in years:
    file_path = os.path.join(base_path, f"votacoes-{year}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, delimiter=';', quotechar='"')
        df["year"] = year  # Add a column to track the year
        dfs.append(df)
    else:
        print(f"File not found: {file_path}")

# Concatenate all dataframes into a single one
df_sessions = pd.concat(dfs, ignore_index=True)

df_sessions

,id,uri,data,dataHoraRegistro,idOrgao,uriOrgao,siglaOrgao,idEvento,uriEvento,aprovacao,...,votosNao,votosOutros,descricao,ultimaAberturaVotacao_dataHoraRegistro,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_dataHoraRegistro,ultimaApresentacaoProposicao_descricao,ultimaApresentacaoProposicao_idProposicao,ultimaApresentacaoProposicao_uriProposicao,year
0,41577-14,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-01-29,NaN,4,https://dadosabertos.camara.leg.br/api/v2/orga...,MESA,0,NaN,1.0,...,0,0,"Aprovação unânime do parecer da Relatora, Dep....",NaN,NaN,NaN,NaN,0,NaN,2003
1,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,NaN,180,https://dadosabertos.camara.leg.br/api/v2/orga...,PLEN,3282,https://dadosabertos.camara.leg.br/api/v2/even...,1.0,...,0,0,Aprovada a Redação Final oferecida pelo Relato...,NaN,Votação da Redação Final.,NaN,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",104848,https://dadosabertos.camara.leg.br/api/v2/prop...,2003
2,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,NaN,180,https://dadosabertos.camara.leg.br/api/v2/orga...,PLEN,3282,https://dadosabertos.camara.leg.br/api/v2/even...,1.0,...,0,0,Aprovada a Redação Final oferecida pelo Relato...,NaN,Votação da Redação Final,NaN,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",104779,https://dadosabertos.camara.leg.br/api/v2/prop...,2003
3,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,NaN,180,https://dadosabertos.camara.leg.br/api/v2/orga...,PLEN,0,NaN,1.0,...,0,0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,NaN,2011-01-20T16:13:25,Apresentação do Requerimento pelo Deputado Wal...,104615,https://dadosabertos.camara.leg.br/api/v2/prop...,2003
4,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,NaN,180,https://dadosabertos.camara.leg.br/api/v2/orga...,PLEN,3282,https://dadosabertos.camara.leg.br/api/v2/even...,1.0,...,0,0,Aprovado o Requerimento,NaN,NaN,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,0,NaN,2003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155022,2266496-54,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,NaN,2003,https://dadosabertos.camara.leg.br/api/v2/orga...,CCJC,0,NaN,0.0,...,0,0,Apresentação do VTS n. 1 CCJC (Voto em Separad...,NaN,NaN,2024-09-25T13:36:14,"Parecer do Relator, Dep. Gilson Marques (NOVO-...",0,NaN,2024
155023,2275427-36,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,NaN,2014,https://dadosabertos.camara.leg.br/api/v2/orga...,CSAUDE,0,NaN,0.0,...,0,0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,NaN,2023-08-09T12:38:28,Apresentação do RPD n. 1 CSAUDE (Requerimento ...,2376653,https://dadosabertos.camara.leg.br/api/v2/prop...,2024
155024,2455796-34,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,NaN,2014,https://dadosabertos.camara.leg.br/api/v2/orga...,CSAUDE,0,NaN,0.0,...,0,0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,NaN,2024-11-21T13:49:49,"Parecer da Relatora, Dep. Ana Pimentel (PT-MG)...",2470893,https://dadosabertos.camara.leg.br/api/v2/prop...,2024
155025,2454233-21,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,NaN,536996,https://dadosabertos.camara.leg.br/api/v2/orga...,CCULT,0,NaN,0.0,...,0,0,Apresentação do VTS n. 1 CCULT (Voto em Separa...,NaN,NaN,2024-11-19T09:21:26,"Parecer do Relator, Dep. Douglas Viegas (UNIÃO...",2470112,https://dadosabertos.camara.leg.br/api/v2/prop...,2024


In [4]:
df_session_selected = df_sessions.drop(columns=(['dataHoraRegistro', 'uriOrgao', 'idEvento', 'uriEvento', 'votosSim', 'votosNao', 'votosOutros', 'ultimaAberturaVotacao_dataHoraRegistro', 'ultimaApresentacaoProposicao_dataHoraRegistro', 'ultimaApresentacaoProposicao_idProposicao', 'ultimaApresentacaoProposicao_uriProposicao']))

df_session_selected

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year
0,41577-14,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-01-29,4,MESA,1.0,"Aprovação unânime do parecer da Relatora, Dep....",NaN,NaN,2003
1,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003
2,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003
3,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003
4,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003
...,...,...,...,...,...,...,...,...,...,...
155022,2266496-54,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2003,CCJC,0.0,Apresentação do VTS n. 1 CCJC (Voto em Separad...,NaN,"Parecer do Relator, Dep. Gilson Marques (NOVO-...",2024
155023,2275427-36,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,Apresentação do RPD n. 1 CSAUDE (Requerimento ...,2024
155024,2455796-34,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,"Parecer da Relatora, Dep. Ana Pimentel (PT-MG)...",2024
155025,2454233-21,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,536996,CCULT,0.0,Apresentação do VTS n. 1 CCULT (Voto em Separa...,NaN,"Parecer do Relator, Dep. Douglas Viegas (UNIÃO...",2024


In [5]:
# Define base path
base_path = '../data/voting/proposition'

# Initialize an empty list to store dataframes
dfs = []

# Loop through years and load each CSV
for year in range(2003, 2025):
    file_path = os.path.join(base_path, f'votacoesProposicoes-{year}.csv')
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, dtype={'idVotacao': str, 'proposicao_id': str}, delimiter=';', quotechar='"')
        df["year"] = year  # Add a column to track the year
        dfs.append(df)
    else:
        print(f"File not found: {file_path}")

# Concatenate all dataframes into a single one
df_propositions = pd.concat(dfs, ignore_index=True)

In [6]:
df_propositions

,idVotacao,uriVotacao,data,descricao,proposicao_id,proposicao_uri,proposicao_titulo,proposicao_ementa,proposicao_codTipo,proposicao_siglaTipo,proposicao_numero,proposicao_ano,year
0,100011-16,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-04-15,Aprovado por Unanimidade o Parecer,100011,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 7412/2002,Dispõe sobre a obrigatoriedade de as fábricas ...,139,PL,7412,2002.0,2003
1,100011-27,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-06-11,Aprovado por Unanimidade o Parecer,100011,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 7412/2002,Dispõe sobre a obrigatoriedade de as fábricas ...,139,PL,7412,2002.0,2003
2,100026-25,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-10-08,Rejeitado o Parecer contra o voto do Deputado ...,100026,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 7414/2002,Dispõe sobre o trabalho escolar de estudantes...,139,PL,7414,2002.0,2003
3,100026-28,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-10-08,Aprovado o Parecer Vencedor contra o voto do D...,100026,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 7414/2002,Dispõe sobre o trabalho escolar de estudantes...,139,PL,7414,2002.0,2003
4,100402-20,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-10-22,Aprovado por Unanimidade o Parecer,100402,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 7425/2002,Dispõe sobre a proibição de comercialização de...,139,PL,7425,2002.0,2003
...,...,...,...,...,...,...,...,...,...,...,...,...,...
114038,947761-37,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-11-13,Alteração do Regime de Tramitação desta propos...,947761,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 410/2015,Cria cadastro nacional de doadores de pele.,139,PL,410,2015.0,2024
114039,947849-23,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-09,"Aprovado o requerimento nº 6903/2017,do Sr. Pa...",947849,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 440/2015,Altera a Lei nº 10.826 de 22 de dezembro de 20...,139,PL,440,2015.0,2024
114040,947849-24,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-09,Alteração do Regime de Tramitação desta propos...,947849,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 440/2015,Altera a Lei nº 10.826 de 22 de dezembro de 20...,139,PL,440,2015.0,2024
114041,994689-78,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-06-19,Aprovado o Parecer.,994689,https://dadosabertos.camara.leg.br/api/v2/prop...,PL 646/2015,"Altera a Lei nº 8.560, de 29 de Dezembro de 19...",139,PL,646,2015.0,2024


In [7]:
df_propositions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114043 entries, 0 to 114042
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   idVotacao             114043 non-null  object 
 1   uriVotacao            114043 non-null  object 
 2   data                  114043 non-null  object 
 3   descricao             114043 non-null  object 
 4   proposicao_id         114043 non-null  object 
 5   proposicao_uri        114043 non-null  object 
 6   proposicao_titulo     114043 non-null  object 
 7   proposicao_ementa     114026 non-null  object 
 8   proposicao_codTipo    114043 non-null  int64  
 9   proposicao_siglaTipo  114043 non-null  object 
 10  proposicao_numero     114043 non-null  int64  
 11  proposicao_ano        113977 non-null  float64
 12  year                  114043 non-null  int64  
dtypes: float64(1), int64(3), object(9)
memory usage: 11.3+ MB


In [8]:
df_propositions['proposicao_id'] = df_propositions['proposicao_id'].astype('int64')

In [9]:
df_propositions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114043 entries, 0 to 114042
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   idVotacao             114043 non-null  object 
 1   uriVotacao            114043 non-null  object 
 2   data                  114043 non-null  object 
 3   descricao             114043 non-null  object 
 4   proposicao_id         114043 non-null  int64  
 5   proposicao_uri        114043 non-null  object 
 6   proposicao_titulo     114043 non-null  object 
 7   proposicao_ementa     114026 non-null  object 
 8   proposicao_codTipo    114043 non-null  int64  
 9   proposicao_siglaTipo  114043 non-null  object 
 10  proposicao_numero     114043 non-null  int64  
 11  proposicao_ano        113977 non-null  float64
 12  year                  114043 non-null  int64  
dtypes: float64(1), int64(4), object(8)
memory usage: 11.3+ MB


In [10]:
# Merge to add 'propositionID' column
df_session_selected = df_session_selected.merge(
    df_propositions[['idVotacao', 'proposicao_id']],
    left_on='id',
    right_on='idVotacao',
    how='left'
)

# Rename column
df_session_selected.rename(columns={'proposicao_id': 'propositionID'}, inplace=True)

# Drop redundant column
df_session_selected.drop(columns=['idVotacao'], inplace=True)

In [11]:
df_session_selected

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,propositionID
0,41577-14,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-01-29,4,MESA,1.0,"Aprovação unânime do parecer da Relatora, Dep....",NaN,NaN,2003,NaN
1,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,96076.0
2,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003,98922.0
3,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003,102277.0
4,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003,102277.0
...,...,...,...,...,...,...,...,...,...,...,...
157155,2266496-54,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2003,CCJC,0.0,Apresentação do VTS n. 1 CCJC (Voto em Separad...,NaN,"Parecer do Relator, Dep. Gilson Marques (NOVO-...",2024,2266496.0
157156,2275427-36,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,Apresentação do RPD n. 1 CSAUDE (Requerimento ...,2024,2275427.0
157157,2455796-34,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,"Parecer da Relatora, Dep. Ana Pimentel (PT-MG)...",2024,2455796.0
157158,2454233-21,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,536996,CCULT,0.0,Apresentação do VTS n. 1 CCULT (Voto em Separa...,NaN,"Parecer do Relator, Dep. Douglas Viegas (UNIÃO...",2024,2454233.0


In [12]:
# Define base path
base_path = '../data/voting/orientations'

# Initialize an empty list to store dataframes
dfs = []

# Loop through years and load each CSV
for year in range(2003, 2025):
    file_path = os.path.join(base_path, f'votacoesOrientacoes-{year}.csv')
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, dtype={'idVotacao': str}, delimiter=';', quotechar='"')
        df["year"] = year  # Add a column to track the year
        dfs.append(df)
    else:
        print(f"File not found: {file_path}")

# Concatenate all dataframes into a single one
df_orientations = pd.concat(dfs, ignore_index=True)

In [13]:
df_orientations

,idVotacao,uriVotacao,siglaOrgao,descricao,siglaBancada,uriBancada,orientacao,year
0,140406-31,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Rejeitado o Requerimento. Sim: 3; Não: 258; Ab...,PSB,https://dadosabertos.camara.leg.br/api/v2/part...,Não,2003
1,140406-31,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Rejeitado o Requerimento. Sim: 3; Não: 258; Ab...,PP,https://dadosabertos.camara.leg.br/api/v2/part...,Não,2003
2,140406-31,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Rejeitado o Requerimento. Sim: 3; Não: 258; Ab...,PV,https://dadosabertos.camara.leg.br/api/v2/part...,Não,2003
3,140406-31,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Rejeitado o Requerimento. Sim: 3; Não: 258; Ab...,PMDB,https://dadosabertos.camara.leg.br/api/v2/part...,Não,2003
4,140406-31,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Rejeitado o Requerimento. Sim: 3; Não: 258; Ab...,PSDB,https://dadosabertos.camara.leg.br/api/v2/part...,Obstrução,2003
...,...,...,...,...,...,...,...,...
94432,2367548-7,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Aprovado o Requerimento de Urgência (Art. 155 ...,Oposição,NaN,NaN,2024
94433,2367548-7,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Aprovado o Requerimento de Urgência (Art. 155 ...,Minoria,NaN,Sim,2024
94434,2367548-7,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Aprovado o Requerimento de Urgência (Art. 155 ...,Novo,https://dadosabertos.camara.leg.br/api/v2/part...,Sim,2024
94435,2367548-7,https://dadosabertos.camara.leg.br/api/v2/vota...,PLEN,Aprovado o Requerimento de Urgência (Art. 155 ...,Bl MdbPsdRepPode,NaN,Sim,2024


In [14]:
# Get unique values from 'siglaBancada'
unique_siglaBancada = df_orientations["siglaBancada"].unique()

# Pivot the df_orientations to have 'siglaBancada' values as columns
df_orientations_pivot = df_orientations.pivot(index='idVotacao', columns='siglaBancada', values='orientacao')
df_orientations_pivot.reset_index(inplace=True)

# Merge with df_sessions_selected
df_session_orientation = df_session_selected.merge(df_orientations_pivot, left_on='id', right_on='idVotacao', how='left')

In [15]:
column_name = "União"  # Replace with the actual column name
unique_values = df_session_orientation[column_name].unique()

print(unique_values)

[nan 'Não' 'Sim' 'Liberado']


In [16]:
len(unique_siglaBancada)

149

In [17]:
# Define mapping function
def map_orientation(value):
    if isinstance(value, str):
        value_lower = value.lower()
        if value_lower in ["sim", "yes", "y"]:
            return 1
        elif value_lower in ["não", "nao", "no", "n"]:
            return -1
    return 0

# Apply mapping to all new columns from siglaBancada
for column in unique_siglaBancada:
    df_session_orientation[column] = df_session_orientation[column].map(map_orientation)

In [18]:
df_session_orientation

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,REDE,REPUBLICANOS,Rede,Republican,SD,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União
0,41577-14,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-01-29,4,MESA,1.0,"Aprovação unânime do parecer da Relatora, Dep....",NaN,NaN,2003,...,0,0,0,0,0,0,0,0,0,0
1,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,0,0,0
2,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003,...,0,0,0,0,0,0,0,0,0,0
3,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003,...,0,0,0,0,0,0,0,0,0,0
4,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157155,2266496-54,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2003,CCJC,0.0,Apresentação do VTS n. 1 CCJC (Voto em Separad...,NaN,"Parecer do Relator, Dep. Gilson Marques (NOVO-...",2024,...,0,0,0,0,0,0,0,0,0,0
157156,2275427-36,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,Apresentação do RPD n. 1 CSAUDE (Requerimento ...,2024,...,0,0,0,0,0,0,0,0,0,0
157157,2455796-34,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,"Parecer da Relatora, Dep. Ana Pimentel (PT-MG)...",2024,...,0,0,0,0,0,0,0,0,0,0
157158,2454233-21,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,536996,CCULT,0.0,Apresentação do VTS n. 1 CCULT (Voto em Separa...,NaN,"Parecer do Relator, Dep. Douglas Viegas (UNIÃO...",2024,...,0,0,0,0,0,0,0,0,0,0


In [19]:
column_name = "União"  # Replace with the actual column name
unique_values = df_session_orientation[column_name].unique()

print(unique_values)

[ 0 -1  1]


In [20]:
# Load authors data
dfs_authors = []
author_years = range(2000, 2025)

type_path = "../data/authors/"
for year in author_years:
    file_path = os.path.join(type_path, f"proposicoesAutores-{year}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, delimiter=';', quotechar='"')
        df["year"] = year  # Add a column to track the year
        dfs_authors.append(df)
    else:
        print(f"File not found: {file_path}")

# Concatenate all dataframes into a single one
df_authors = pd.concat(dfs_authors, ignore_index=True)

In [21]:
df_authors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1593168 entries, 0 to 1593167
Data columns (total 13 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   idProposicao       1593168 non-null  int64  
 1   uriProposicao      1593168 non-null  object 
 2   idDeputadoAutor    1444263 non-null  float64
 3   uriAutor           1583113 non-null  object 
 4   codTipoAutor       1593168 non-null  int64  
 5   tipoAutor          1593168 non-null  object 
 6   nomeAutor          1593168 non-null  object 
 7   siglaPartidoAutor  1457139 non-null  object 
 8   uriPartidoAutor    1404892 non-null  object 
 9   siglaUFAutor       1457085 non-null  object 
 10  ordemAssinatura    1593168 non-null  int64  
 11  proponente         1593168 non-null  int64  
 12  year               1593168 non-null  int64  
dtypes: float64(1), int64(5), object(7)
memory usage: 158.0+ MB


In [22]:
df_session_orientation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157160 entries, 0 to 157159
Columns: 161 entries, id to União
dtypes: float64(2), int64(151), object(8)
memory usage: 193.0+ MB


In [23]:
# Perform the merge
df_session_author = df_session_orientation.merge(
    df_authors[['idProposicao', 'tipoAutor', 'nomeAutor']],
    left_on='propositionID',
    right_on='idProposicao',
    how='left'
)

# Rename new columns
df_session_author.rename(columns={"tipoAutor": "author_type", "nomeAutor": "author"}, inplace=True)

In [24]:
df_session_author

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,Republican,SD,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,idProposicao,author_type,author
0,41577-14,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-01-29,4,MESA,1.0,"Aprovação unânime do parecer da Relatora, Dep....",NaN,NaN,2003,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,96076.0,Órgão do Poder Executivo,Poder Executivo
2,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003,...,0,0,0,0,0,0,0,98922.0,Órgão do Poder Executivo,Poder Executivo
3,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003,...,0,0,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro
4,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003,...,0,0,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386575,2266496-54,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2003,CCJC,0.0,Apresentação do VTS n. 1 CCJC (Voto em Separad...,NaN,"Parecer do Relator, Dep. Gilson Marques (NOVO-...",2024,...,0,0,0,0,0,0,0,2266496.0,Deputado(a),Marcelo Ramos
386576,2275427-36,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,Apresentação do RPD n. 1 CSAUDE (Requerimento ...,2024,...,0,0,0,0,0,0,0,2275427.0,Deputado(a),Júlio Delgado
386577,2455796-34,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,"Parecer da Relatora, Dep. Ana Pimentel (PT-MG)...",2024,...,0,0,0,0,0,0,0,2455796.0,Deputado(a),Carla Ayres
386578,2454233-21,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,536996,CCULT,0.0,Apresentação do VTS n. 1 CCULT (Voto em Separa...,NaN,"Parecer do Relator, Dep. Douglas Viegas (UNIÃO...",2024,...,0,0,0,0,0,0,0,2454233.0,Deputado(a),Defensor Stélio Dener


In [25]:
# Drop rows where 'ultimaApresentacaoProposicao_idProposicao' or 'author' is NaN, null, or zero
df_session_author = df_session_author.dropna(subset=["idProposicao", "author"])
df_session_author = df_session_author[(df_session_author["idProposicao"] != 0) & (df_session_author["author"] != 0)]

In [26]:
df_session_author

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,Republican,SD,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,idProposicao,author_type,author
1,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,96076.0,Órgão do Poder Executivo,Poder Executivo
2,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003,...,0,0,0,0,0,0,0,98922.0,Órgão do Poder Executivo,Poder Executivo
3,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003,...,0,0,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro
4,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003,...,0,0,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro
5,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,96076.0,Órgão do Poder Executivo,Poder Executivo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386575,2266496-54,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2003,CCJC,0.0,Apresentação do VTS n. 1 CCJC (Voto em Separad...,NaN,"Parecer do Relator, Dep. Gilson Marques (NOVO-...",2024,...,0,0,0,0,0,0,0,2266496.0,Deputado(a),Marcelo Ramos
386576,2275427-36,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,Apresentação do RPD n. 1 CSAUDE (Requerimento ...,2024,...,0,0,0,0,0,0,0,2275427.0,Deputado(a),Júlio Delgado
386577,2455796-34,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,2014,CSAUDE,0.0,Apresentação do VTS n. 1 CSAUDE (Voto em Separ...,NaN,"Parecer da Relatora, Dep. Ana Pimentel (PT-MG)...",2024,...,0,0,0,0,0,0,0,2455796.0,Deputado(a),Carla Ayres
386578,2454233-21,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,536996,CCULT,0.0,Apresentação do VTS n. 1 CCULT (Voto em Separa...,NaN,"Parecer do Relator, Dep. Douglas Viegas (UNIÃO...",2024,...,0,0,0,0,0,0,0,2454233.0,Deputado(a),Defensor Stélio Dener


In [27]:
df_session_PLEN = df_session_author[(df_session_author["idOrgao"] == 180)]

In [28]:
df_session_PLEN

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,Republican,SD,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,idProposicao,author_type,author
1,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,96076.0,Órgão do Poder Executivo,Poder Executivo
2,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003,...,0,0,0,0,0,0,0,98922.0,Órgão do Poder Executivo,Poder Executivo
3,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003,...,0,0,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro
4,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003,...,0,0,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro
5,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,96076.0,Órgão do Poder Executivo,Poder Executivo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386569,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,0,0,2473375.0,Deputado(a),Doutor Luizinho
386570,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,0,0,2473375.0,Deputado(a),Márcio Jerry
386571,2460010-35,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,0.0,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,NaN,NaN,2024,...,0,0,0,0,0,0,0,2460010.0,Deputado(a),José Guimarães
386572,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,0,0,2460010.0,Deputado(a),José Guimarães


In [29]:
# Load classified voting texts data
dfs_types = []
types_path = "../data/classified_llm_datasets/"

i = 1
while True:
    file_path = os.path.join(types_path, f"classified_voting_texts_batch_{i}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        dfs_types.append(df)
        i += 1
    else:
        break

# Concatenate all classified voting datasets into one
df_types = pd.concat(dfs_types, ignore_index=True)

In [30]:
df_types

,idVotacao,siglaOrgao,aprovacao,votosSim,votosNao,votosOutros,descricao,year,nomeOrgao,tipoOrgao,proposicao_id,proposicao_titulo,proposicao_ementa,proposicao_siglaTipo,proposicao_ano,proposicao_ano_arquivo,has_votes,category,detected_phrase,classification
0,96076-49,PLEN,1,0,0,0,Aprovada a Redação Final oferecida pelo Relato...,2003,Plenário,Plenário Virtual,96076.0,PL 7262/2002,Dispõe sobre o Estatuto de Defesa do Torcedor ...,PL,2002.0,2003.0,False,Proposition Approval,aprovada a redação,Approval of Final Wording
1,96076-46,PLEN,1,0,0,0,Aprovado o Substitutivo oferecido pelo Relator...,2003,Plenário,Plenário Virtual,96076.0,PL 7262/2002,Dispõe sobre o Estatuto de Defesa do Torcedor ...,PL,2002.0,2003.0,False,Text Editing,aprovado o substitutivo,Approval of Substitutes
2,19408-72,PLEN,1,0,0,0,Aprovado o Requerimento do Sr. Dep. José Carlo...,2003,Plenário,Plenário Virtual,19408.0,PL 3285/1992,Dispõe sobre a utilização e proteção da Mata A...,PL,1992.0,2003.0,False,Change in Agenda,retirada de pauta,Approval of Requests
3,21283-45,PLEN,1,0,0,0,Aprovado o Requerimento do Sr. Dep. Mendes Rib...,2003,Plenário,Plenário Virtual,21283.0,PL 6132/1990,Dispõe sobre o registro de pessoas fisicas ou ...,PL,1990.0,2003.0,False,Change in Agenda,retirada de pauta,Approval of Requests
4,90170-30,PLEN,1,0,0,0,"Aprovado requerimento do Líderes que requer, n...",2003,Plenário,Plenário Virtual,90170.0,PL 7241/2002,Dispõe sobre a alienação por doação de uma Cor...,PL,2002.0,2003.0,False,Other,NaN,Approval of Requests
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8774,2473375-78,PLEN,1,277,174,2,Mantido o texto. Sim: 277; Não: 174; Abstenção...,2024,Plenário,Plenário Virtual,2473375.0,PL 4614/2024,"Altera a Lei nº 8.171, de 17 de janeiro de 199...",PL,2024.0,2024.0,True,Proposition Approval,mantido o texto,Approval of Final Wording
8775,2473375-80,PLEN,1,0,0,0,Aprovada a Redação Final assinada pelo relator...,2024,Plenário,Plenário Virtual,2473375.0,PL 4614/2024,"Altera a Lei nº 8.171, de 17 de janeiro de 199...",PL,2024.0,2024.0,False,Proposition Approval,aprovada a redação,Approval of Final Wording
8776,2460010-35,PLEN,0,82,193,1,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,2024,Plenário,Plenário Virtual,2460010.0,PL 3802/2024,"Altera a Lei nº 14.467, de 16 de novembro de 2...",PL,2024.0,2024.0,True,Other,NaN,Approval of Requests
8777,2460010-43,PLEN,1,340,117,2,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",2024,Plenário,Plenário Virtual,2460010.0,PL 3802/2024,"Altera a Lei nº 14.467, de 16 de novembro de 2...",PL,2024.0,2024.0,True,Proposition Approval,aprovado o projeto,Approval of Final Wording


In [31]:
# Merge classification data with df_session_PLEN
df_session_types = df_session_PLEN.merge(df_types[['idVotacao', 'classification']],
                                            left_on='id', right_on='idVotacao',
                                            how='left')

# Rename new column
df_session_types.rename(columns={"classification": "session_Type"}, inplace=True)

In [32]:
df_session_types

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,idProposicao,author_type,author,idVotacao_y,session_Type
0,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,96076.0,Órgão do Poder Executivo,Poder Executivo,96076-49,Approval of Final Wording
1,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003,...,0,0,0,0,0,98922.0,Órgão do Poder Executivo,Poder Executivo,NaN,NaN
2,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003,...,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro,NaN,NaN
3,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003,...,0,0,0,0,0,102277.0,Deputado(a),Walter Pinheiro,NaN,NaN
4,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,96076.0,Órgão do Poder Executivo,Poder Executivo,96076-46,Approval of Substitutes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62477,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,2473375.0,Deputado(a),Doutor Luizinho,2473375-80,Approval of Final Wording
62478,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,2473375.0,Deputado(a),Márcio Jerry,2473375-80,Approval of Final Wording
62479,2460010-35,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,0.0,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,NaN,NaN,2024,...,0,0,0,0,0,2460010.0,Deputado(a),José Guimarães,2460010-35,Approval of Requests
62480,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,2460010.0,Deputado(a),José Guimarães,2460010-43,Approval of Final Wording


In [33]:
df_session_types = df_session_types.drop(columns=['idProposicao', 'idVotacao_y'])

In [34]:
df_session_types

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,Republican,SD,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,author_type,author,session_Type
0,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording
1,98922-29,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Ronaldo Ca...",2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,NaN
2,102277-6,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado requerimento n. 264/2002 de Líderes q...,NaN,Apresentação do Requerimento pelo Deputado Wal...,2003,...,0,0,0,0,0,0,0,Deputado(a),Walter Pinheiro,NaN
3,103255-3,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Requerimento,NaN,Apresentação da Urgência (Art. 155 do RICD) pe...,2003,...,0,0,0,0,0,0,0,Deputado(a),Walter Pinheiro,NaN
4,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Substitutes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62477,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,0,0,Deputado(a),Doutor Luizinho,Approval of Final Wording
62478,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,0,0,Deputado(a),Márcio Jerry,Approval of Final Wording
62479,2460010-35,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,0.0,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,NaN,NaN,2024,...,0,0,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Requests
62480,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording


In [35]:
df_session_types = df_session_types.dropna(subset=["session_Type"])

In [36]:
df_session_types

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,Republican,SD,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,author_type,author,session_Type
0,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording
4,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Substitutes
20,90170-30,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,"Aprovado requerimento do Líderes que requer, n...",NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Requests
30,101651-2,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,Aprovado,NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording
34,90170-25,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-27,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Agnaldo Mu...",2003,...,0,0,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62477,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,0,0,Deputado(a),Doutor Luizinho,Approval of Final Wording
62478,2473375-80,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,Parecer às Emendas de Plenário proferido pelo ...,2024,...,0,0,0,0,0,0,0,Deputado(a),Márcio Jerry,Approval of Final Wording
62479,2460010-35,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,0.0,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,NaN,NaN,2024,...,0,0,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Requests
62480,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording


In [37]:
# Load propositions themes data
dfs_themes = []
themes_path = "../data/propositions/"
for year in range(2000, 2025):
    file_path = os.path.join(themes_path, f"proposicoesTemas-{year}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, delimiter=';', quotechar='"')
        dfs_themes.append(df)
    else:
        print(f"File not found: {file_path}")

In [38]:
# Concatenate all themes dataframes into one
df_themes = pd.concat(dfs_themes, ignore_index=True)

# Extract proposition code from 'uriProposicao'
df_themes['idProposicao'] = df_themes['uriProposicao'].apply(lambda x: int(re.search(r'\d+$', x).group()) if isinstance(x, str) and re.search(r'\d+$', x) else None)

# Merge themes data with df_session_PLEN
df_session_theme = df_session_types.merge(df_themes[['idProposicao', 'tema']],
                                         left_on='propositionID',
                                         right_on='idProposicao',
                                         how='left')

# Rename new column
df_session_theme.rename(columns={"tema": "theme"}, inplace=True)

In [39]:
df_session_theme

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,author_type,author,session_Type,idProposicao,theme
0,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,96076.0,Esporte e Lazer
1,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Substitutes,96076.0,Esporte e Lazer
2,90170-30,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,"Aprovado requerimento do Líderes que requer, n...",NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Requests,90170.0,Defesa e Segurança
3,101651-2,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,Aprovado,NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,90170.0,Defesa e Segurança
4,90170-25,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-27,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Agnaldo Mu...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,90170.0,Defesa e Segurança
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39065,2460010-35,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,0.0,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,NaN,NaN,2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Requests,2460010.0,Finanças Públicas e Orçamento
39066,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Economia
39067,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Finanças Públicas e Orçamento
39068,2460010-45,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Economia


In [40]:
df_session_theme = df_session_theme.dropna(subset=["theme"])

In [41]:
df_session_theme

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,author_type,author,session_Type,idProposicao,theme
0,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,96076.0,Esporte e Lazer
1,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Substitutes,96076.0,Esporte e Lazer
2,90170-30,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,"Aprovado requerimento do Líderes que requer, n...",NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Requests,90170.0,Defesa e Segurança
3,101651-2,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,Aprovado,NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,90170.0,Defesa e Segurança
4,90170-25,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-27,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Agnaldo Mu...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,90170.0,Defesa e Segurança
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39065,2460010-35,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,0.0,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,NaN,NaN,2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Requests,2460010.0,Finanças Públicas e Orçamento
39066,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Economia
39067,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Finanças Públicas e Orçamento
39068,2460010-45,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Economia


In [43]:
df_all_info = df_session_theme.copy()

In [44]:
df_all_info

,id,uri,data,idOrgao,siglaOrgao,aprovacao,descricao,ultimaAberturaVotacao_descricao,ultimaApresentacaoProposicao_descricao,year,...,SDD,SOLIDARIEDADE,Solidaried,UNIÃO,União,author_type,author,session_Type,idProposicao,theme
0,96076-49,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,96076.0,Esporte e Lazer
1,96076-46,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-19,180,PLEN,1.0,Aprovado o Substitutivo oferecido pelo Relator...,Votação em turno único.,"Parecer Proferido em Plenário, Dep. Luiz Eduar...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Substitutes,96076.0,Esporte e Lazer
2,90170-30,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,"Aprovado requerimento do Líderes que requer, n...",NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Requests,90170.0,Defesa e Segurança
3,101651-2,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-26,180,PLEN,1.0,Aprovado,NaN,Apresentação do Requerimento.,2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,90170.0,Defesa e Segurança
4,90170-25,https://dadosabertos.camara.leg.br/api/v2/vota...,2003-02-27,180,PLEN,1.0,Aprovada a Redação Final oferecida pelo Relato...,Votação da Redação Final,"Parecer Proferido em Plenário, Dep. Agnaldo Mu...",2003,...,0,0,0,0,0,Órgão do Poder Executivo,Poder Executivo,Approval of Final Wording,90170.0,Defesa e Segurança
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39065,2460010-35,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,0.0,Rejeitado o Requerimento. Sim: 82; Não: 193; T...,NaN,NaN,2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Requests,2460010.0,Finanças Públicas e Orçamento
39066,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Economia
39067,2460010-43,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,"Aprovado o Projeto de Lei nº 3.802, de 2024. S...",Votação em turno único.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Finanças Públicas e Orçamento
39068,2460010-45,https://dadosabertos.camara.leg.br/api/v2/vota...,2024-12-19,180,PLEN,1.0,Aprovada a Redação Final assinada pelo relator...,Votação da Redação Final.,"Parecer proferido em Plenário pelo Relator, De...",2024,...,0,0,0,0,0,Deputado(a),José Guimarães,Approval of Final Wording,2460010.0,Economia


In [45]:
df_all_info.to_csv("df_all_info.csv", index=False)

print("DataFrame successfully exported as 'df_all_info.csv'")

DataFrame successfully exported as 'df_all_info.csv'
